1: Cài đặt các thư viện cần thiết

In [ ]:
!pip install transformers datasets sentencepiece accelerate underthesea

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 122.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 88.9 MB/s eta 0:00:00


2: Giải nén dữ liệu

Archive:  colab_dataset.zip
   creating: colab_dataset/
  inflating: colab_dataset/test_data.json  
  inflating: colab_dataset/train_data.json  
  inflating: colab_dataset/valid_data.json  


3: Load dữ liệu và Mô hình

In [ ]:
import json
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

# 1. Load Dataset
def load_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

train_data = load_json('colab_dataset/train_data.json')
valid_data = load_json('colab_dataset/valid_data.json')

# Biến đổi thành HuggingFace Dataset
train_dataset = Dataset.from_list(train_data)
valid_dataset = Dataset.from_list(valid_data)

datasets = DatasetDict({
    "train": train_dataset,
    "validation": valid_dataset
})

# 2. Load Tokenizer & Model (vinai/bartpho-word)
model_checkpoint = "vinai/bartpho-word"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

# Đóng băng Encoder để tăng tốc độ huấn luyện giống bài báo
for param in model.model.encoder.parameters():
    param.requires_grad = False

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/897 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


4: Tiền xử lý dữ liệu (Tokenization)

In [ ]:
max_length = 256

def preprocess_function(examples):
    inputs = [ex for ex in examples["vi"]]
    targets = [ex for ex in examples["ba"]]

    model_inputs = tokenizer(inputs, max_length=max_length, truncation=True)

   # Setup the tokenizer for targets
    labels = tokenizer(text_target=targets, max_length=max_length, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = datasets.map(preprocess_function, batched=True)

Map:   0%|          | 0/9275 [00:00<?, ? examples/s]

Map:   0%|          | 0/1988 [00:00<?, ? examples/s]

5: Cấu hình Huấn Luyện (Training)

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./bartpho-bana-nmt",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=21, # Chỉnh số epoch tuỳ ý
    predict_with_generate=True,
    fp16=True, # Dùng Mixed Precision để train nhanh hơn trên GPU
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
)

# Bắt đầu huấn luyện
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.716677,1.537148
2,1.576780,1.100815
3,1.220911,0.911687
4,0.999657,0.799969
5,0.850318,0.730257
6,0.738083,0.683059
7,0.570959,0.646129
8,0.514931,0.616705
9,0.467665,0.597106
10,0.416618,0.579670


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=12180, training_loss=0.5837911906500756, metrics={'train_runtime': 10871.9386, 'train_samples_per_second': 17.915, 'train_steps_per_second': 1.12, 'total_flos': 3.613182236796518e+16, 'train_loss': 0.5837911906500756, 'epoch': 21.0})

6: Lưu mô hình về máy tính

In [ ]:
# Lưu mô hình
trainer.save_model("./best-bana-model")

# Nén mô hình để tải về
!zip -r best-bana-model.zip best-bana-model/

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  adding: best-bana-model/ (stored 0%)
  adding: best-bana-model/generation_config.json (deflated 41%)
  adding: best-bana-model/training_args.bin (deflated 53%)
  adding: best-bana-model/added_tokens.json (stored 0%)
  adding: best-bana-model/bpe.codes (deflated 59%)
  adding: best-bana-model/model.safetensors (deflated 24%)
  adding: best-bana-model/config.json (deflated 58%)
  adding: best-bana-model/tokenizer_config.json (deflated 76%)
  adding: best-bana-model/vocab.txt (deflated 55%)
